# An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (ViT)

Replication of Dosovitskiy, Beyer, Kolesnikov, Weissenborn, Zhai, Unterthiner, Dehghani,
Minderer, Heigold, Gelly, Uszkoreit and Houlsby (2020), *An Image is Worth 16x16 Words:
Transformers for Image Recognition at Scale* (arXiv:2010.11929).

The Vision Transformer applies a standard Transformer encoder directly to images: the image
is split into fixed-size patches, each patch is linearly embedded into a token, a learnable
class token and positional embeddings are added, and the sequence is processed by a
Transformer. The class token's final representation is classified. We implement this from
scratch and reproduce the result that a pure-attention model, with no convolutions, learns
to classify images, reaching high accuracy on MNIST.

In [1]:
import torch, torch.nn as nn
import torchvision as tv, torchvision.transforms as T
torch.manual_seed(0)

In [2]:
tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train = tv.datasets.MNIST("../data", train=True,  download=True, transform=tf)
test  = tv.datasets.MNIST("../data", train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_dl  = torch.utils.data.DataLoader(test,  batch_size=512)
print("train", len(train), "test", len(test))

train 60000 test 10000


In [3]:
# Split each 28x28 image into 7x7 patches (16 patches) and linearly embed them.
PATCH, DIM, DEPTH, HEADS = 7, 64, 4, 4
N_PATCH = (28 // PATCH) ** 2
class PatchEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(1, DIM, PATCH, stride=PATCH)        # non-overlapping patch projection
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)           # (B, n_patch, DIM)
print("patches per image:", N_PATCH)

patches per image: 16


In [4]:
class ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch = PatchEmbed()
        self.cls = nn.Parameter(torch.zeros(1, 1, DIM))
        self.pos = nn.Parameter(torch.zeros(1, N_PATCH + 1, DIM))
        layer = nn.TransformerEncoderLayer(DIM, HEADS, 4*DIM, batch_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, DEPTH)
        self.norm = nn.LayerNorm(DIM); self.head = nn.Linear(DIM, 10)
    def forward(self, x):
        B = x.size(0)
        tok = torch.cat([self.cls.expand(B, -1, -1), self.patch(x)], 1) + self.pos
        return self.head(self.norm(self.enc(tok))[:, 0])         # classify from the class token

net = ViT()
print("parameters:", sum(p.numel() for p in net.parameters()))

parameters: 205066


In [5]:
opt = torch.optim.AdamW(net.parameters(), lr=3e-4); lf = nn.CrossEntropyLoss()
def evaluate():
    net.eval(); correct = 0
    with torch.no_grad():
        for x, y in test_dl: correct += (net(x).argmax(1) == y).sum().item()
    return correct / len(test)
for ep in range(4):
    net.train()
    for x, y in train_dl:
        opt.zero_grad(); lf(net(x), y).backward(); opt.step()
    print(f"epoch {ep+1}: test_acc={evaluate():.4f}")

epoch 1: test_acc=0.8262


epoch 2: test_acc=0.9128


epoch 3: test_acc=0.9406


epoch 4: test_acc=0.9529


In [6]:
acc = evaluate()
print(f"Final MNIST test accuracy (pure-attention ViT, no convolutions in the encoder): {acc*100:.2f}%")

Final MNIST test accuracy (pure-attention ViT, no convolutions in the encoder): 95.29%
